# E-Commerce Churn, RFM & CLV Preprocessing — Google Colab Runner

This notebook runs the full preprocessing pipeline on the **Olist Brazilian E-Commerce** dataset and produces the 5 CSV files needed to build the Power BI dashboard:

1. `fact_orders.csv`
2. `dim_customers_rfm.csv`
3. `cohort_table.csv`
4. `geo_sales.csv`
5. `kpi_summary.csv`

**Instructions:**
1. Run the cells in order, top to bottom.
2. When prompted in Cell 2, upload the project zip (or the raw Olist Kaggle CSVs / zip).
3. After the pipeline finishes, Cell 5 zips the `output/` folder and downloads it to your computer — load those CSVs straight into Power BI.

## 1. Install dependencies (pandas/numpy ship with Colab, this is just a safety check)

In [ ]:
!pip install -q pandas numpy

## 2. Upload your data
Run this cell, then click **Choose Files** and upload either:
- `archive.zip` (the raw Olist Kaggle download), or
- the individual `olist_*.csv` files

If you already have this project's zip with `data/` pre-populated, you can instead mount Google Drive (see commented-out cell below) and skip the upload.

In [ ]:
import os, zipfile, shutil
from google.colab import files

os.makedirs('data', exist_ok=True)

uploaded = files.upload()  # select archive.zip or the raw CSVs

for fname in uploaded.keys():
    if fname.lower().endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('data')
        print(f'Extracted {fname} into ./data')
    else:
        shutil.move(fname, os.path.join('data', fname))
        print(f'Moved {fname} into ./data')

print('\nFiles now in ./data:')
print(os.listdir('data'))

### (Optional) Mount Google Drive instead of uploading
Uncomment and run if your CSVs already live in Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/olist_data'  # change to your path
# import shutil, os
# for f in os.listdir(DATA_DIR):
#     shutil.copy(os.path.join(DATA_DIR, f), os.path.join('data', f))

## 3. Get the pipeline script
Either upload `preprocess_pipeline.py` (from this project's `src/` folder) alongside this notebook, or paste/run it as a module. The cell below assumes `preprocess_pipeline.py` is in the Colab working directory (`/content/`). If it isn't there yet, upload it now via the Colab file browser on the left, or run the upload cell below.

In [ ]:
import os
if not os.path.exists('preprocess_pipeline.py'):
    from google.colab import files
    print('Please select preprocess_pipeline.py from the project src/ folder')
    up = files.upload()
    assert 'preprocess_pipeline.py' in up, 'preprocess_pipeline.py was not uploaded'
print('Ready: preprocess_pipeline.py found in working directory.')

## 4. Run the preprocessing pipeline

In [ ]:
from preprocess_pipeline import run_pipeline

outputs = run_pipeline(data_dir='data', output_dir='output')

print('\nPreview of KPI summary:')
print(outputs['kpi_summary.csv'])

## 5. (Optional) Quick visual sanity-checks directly in Colab

In [ ]:
import matplotlib.pyplot as plt

rfm = outputs['dim_customers_rfm.csv']
rfm['RFM_segment'].value_counts().plot(kind='barh', title='Customers per RFM segment')
plt.xlabel('Number of customers')
plt.tight_layout()
plt.show()

In [ ]:
geo = outputs['geo_sales.csv']
geo.sort_values('total_sales', ascending=False).head(10).plot(
    x='customer_state', y='total_sales', kind='bar', legend=False,
    title='Top 10 states by total sales')
plt.ylabel('Total sales (R$)')
plt.tight_layout()
plt.show()

## 6. Download the cleaned output files (load these into Power BI)

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('powerbi_ready_data', 'zip', 'output')
files.download('powerbi_ready_data.zip')